In [1]:
%load_ext autoreload
%autoreload 2

# Run this cell first to prevent cached modules!

# RAG Ingestion Pipeline Verification

This notebook verifies the `RAGIngestion` class as required by the Stage 1 Prototyping rules.

In [3]:
import sys
import os
sys.path.append("../../..")
from google.cloud import bigquery
from pipelines.enterprise_knowledge_base.rag_ingestion import RAGIngestion
from pipelines.enterprise_knowledge_base.orchestrator import KBIngestionPipeline

In [4]:
# Initialize the pipeline orchestrator
PROJECT_ID = "ag-core-dev-fdx7"
pipeline = KBIngestionPipeline(project_id=PROJECT_ID)


In [5]:
# Run the full pipeline (staging + vectorization)
GCS_URI = "gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf"

try:
    pipeline.trigger_pipeline(GCS_URI)
except Exception as e:
    print(f'Pipeline execution result: {e}')

Triggering pipeline for gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf...
Successfully staged 40 chunks.
Initiating BigQuery ML vectorization...
Vectorization complete.


In [8]:
# Verify BigQuery: Check if embeddings were generated
bq_client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT chunk_id, gcs_uri, ARRAY_LENGTH(embedding) as embedding_length 
FROM `{PROJECT_ID}.knowledge_base.documents_chunks` 
WHERE gcs_uri = 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf'
LIMIT 5
"""

results = bq_client.query(query).result()
for row in results:
     print(dict(row))

{'chunk_id': '293df889-a968-4ed0-8783-88e05d8637a7', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}
{'chunk_id': '2446c254-cc55-4828-bb58-454f2ad3918c', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}
{'chunk_id': '0e27acec-136b-4777-ac95-032ae01f62a7', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}
{'chunk_id': '17ed1e54-e0f2-4720-a767-fe40d823ea8b', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 1408}
{'chunk_id': '721c1dc0-259a-49dc-8e48-0c3efbe72aa3', 'gcs_uri': 'gs://kb-landing-zone-test/ingested/Gumtree - SOW WIP.pdf', 'embedding_length': 0}
